# Score Calibration, Drift Detection, and Significance Testing for Retrieval

The narrative companion to `significance_testing_calibration.py`, the canonical reference that
**owns every number** this topic reports. We never redefine the math here — we import the module and
walk it section by section. Its prerequisites (`set-metrics-precision-recall-map-mrr` and
`ndcg-discount-geometry`) framed a metric as an **estimator** and left one question open: *is an
observed gap real?* This topic closes it with a **paired** significance test, then generalizes the
same two-distribution comparison to **calibration** (score vs truth) and **drift** (now vs then).

In [ ]:
import pathlib, sys

# Path-robust import: find the module regardless of the kernel's working directory. The module itself
# adds every ancestor notebook dir (the multi-vector subtree + both eval prereqs) to sys.path.
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "significance-testing-calibration",
              pathlib.Path("notebooks/significance-testing-calibration")):
    if (_cand / "significance_testing_calibration.py").exists():
        sys.path.insert(0, str(_cand.resolve()))
        break

import numpy as np
import significance_testing_calibration as S
from significance_testing_calibration import (
    get_corpus, ap_samples, ndcg_samples, paired_diffs, paired_t_test,
    permutation_test, paired_bootstrap_test, pairwise_tests, holm_bonferroni, bh_fdr,
    power_required_n, paired_separation_n, ece_table, ranking_preserved_platt,
    brier_decomposition, leg_scores, platt_scale, apply_platt,
    drift_summary, silent_decay, input_vs_outcome_drift, WORKED_PAIR,
)
from set_metrics_precision_recall_map_mrr import projected_separation_n
from ndcg_discount_geometry import projected_ndcg_separation_n
corpus = get_corpus()
print("corpus:", corpus["n_queries"], "queries x", corpus["n_docs"], "docs; worked pair =", WORKED_PAIR)

## 1. The paired test — pairing cancels per-query difficulty

The prerequisites compared two legs' **means** with independent confidence intervals, a conservative
read because both legs run on the *same* queries. We difference query-by-query: `d_q = m_A(q) - m_B(q)`.
Because easy queries are easy for every leg, `var(d) = var(A) + var(B) - 2 cov(A,B)` is well below the
unpaired sum, so the paired interval excludes 0 at far fewer queries.

In [ ]:
a, b = WORKED_PAIR
for metric, samp in (("MAP", ap_samples), ("NDCG", ndcg_samples)):
    d = paired_diffs(samp(corpus, a), samp(corpus, b))
    tt = paired_t_test(d)
    unp = (projected_separation_n(corpus, a, b, "qrels_set", n_max=100000) if metric == "MAP"
           else projected_ndcg_separation_n(corpus, a, b, n_max=100000))
    print(f"{metric:5s} {a} vs {b}: mean_d={tt['mean']:+.4f} t={tt['t']:.2f} p={tt['p']:.2e} "
          f"d_z={tt['cohen_dz']:.3f}")
    print(f"      paired separates at n={paired_separation_n(d)}  unpaired needs n={unp}  "
          f"80%-power n={power_required_n(d)}")

**The cliffhanger closed.** For the closest NDCG pair the crude overlapping-interval read asked for
~185 queries; the rigorous 80%-power count is **116**, and the single-realization "interval clears 0"
count is **57** (a one-time event at ~50% power, hence below the power count). And pairing *tightens*
without *manufacturing*: the MAP gap is significant at n=40 (p ≈ 4e-4) while the NDCG gap is **not yet**
(p ≈ 0.107) — the correct answer to "is the 0.05 NDCG gap real?" is *not on this evidence*.

In [ ]:
for metric, samp in (("MAP", ap_samples), ("NDCG", ndcg_samples)):
    sa, sb = samp(corpus, a), samp(corpus, b)
    d = paired_diffs(sa, sb)
    ratio = np.var(d, ddof=1) / (np.var(sa, ddof=1) + np.var(sb, ddof=1))
    print(f"{metric:5s} corr(A,B)={np.corrcoef(sa, sb)[0,1]:.3f}  var(d)/(varA+varB)={ratio:.3f}  "
          f"CI excludes 0 <=> reject (duality): {(paired_t_test(d)['ci_hi'] < 0) == (paired_t_test(d)['p'] < 0.05)}")

## 2. Distribution-free tests and the cost of three comparisons

The per-query differences are bounded and skewed, so the t-test's normality is approximate. The
**permutation** (sign-flip) test is exact under exchangeability; the **bootstrap** assumes neither.
Three legs is `C(3,2)=3` simultaneous tests, so we correct (Bonferroni / Holm / BH).

In [ ]:
d = paired_diffs(ap_samples(corpus, a), ap_samples(corpus, b))
print(f"worked MAP pair:  t_p={paired_t_test(d)['p']:.2e}  perm_p={permutation_test(d)['p']:.2e}  "
      f"boot_p={paired_bootstrap_test(d)['p']:.2e}  (all agree)")
print()
rows = pairwise_tests(corpus, "ndcg", "paired_t")
pv = [r["p"] for r in rows]
hb, fdr = holm_bonferroni(pv), bh_fdr(pv)
for i, r in enumerate(rows):
    print(f"  {r['pair'][0]:16s} vs {r['pair'][1]:16s} raw={pv[i]:.2e} "
          f"bonf={hb['bonferroni'][i]:.3f} holm={hb['holm'][i]:.3f} bh={fdr['bh'][i]:.3f} "
          f"reject_bonf={hb['reject_bonf'][i]}")

## 3. Calibration — is a score a probability?

Pool every (query, doc) score and its relevance label. A reliability diagram plots empirical relevance
against confidence; **ECE** is the area off the diagonal. Raw cosine/MaxSim scores are wildly
over-confident — which is why rank fusion fuses *ranks*, not scores. **Platt** and **isotonic**
recalibration crush ECE *without changing the ranking* (AUC identical): calibration is orthogonal to
ranking.

In [ ]:
s0, y0, _ = leg_scores(corpus, "dense")
print(f"pooled pairs={s0.size}  positives={int(y0.sum())}  base rate={y0.mean():.4f}\n")
for leg in S.LEG_NAMES:
    t = ece_table(corpus, leg)
    print(f"{leg:16s} ECE raw={t['raw']['ece']:.4f} -> platt={t['platt']['ece']:.4f} "
          f"iso={t['isotonic']['ece']:.4f}   AUC raw={t['auc_raw']:.6f} == platt={t['auc_platt']:.6f}  "
          f"ranking preserved={ranking_preserved_platt(corpus, leg, t['platt_params']['a'], t['platt_params']['b'])}")
bd = brier_decomposition(apply_platt(s0, *platt_scale(s0, y0)), y0)
print(f"\ndense Brier decomp: reliability={bd['reliability']:.5f} resolution={bd['resolution']:.5f} "
      f"uncertainty={bd['uncertainty']:.5f}  (rel - res + unc == binned Brier)")

## 4. Drift — has the distribution moved?

Drift is a two-sample test of a reference window against a current one. The **KS** statistic is the
sup-gap between empirical CDFs; **PSI** is a symmetrized KL (the credit-risk traffic light). Two
lessons: a *silent* decay the aggregate mean misses is caught by the **paired** test; and a pure
*covariate shift* fires the input alarm with **no** quality loss — input drift alone cannot diagnose
decay.

In [ ]:
print("degradation knob (sigma):")
for r in drift_summary(corpus):
    print(f"  sigma={r['sigma']:.2f}  PSI={r['psi']:.3f} ({r['light']})  KS_D={r['ks_stat']:.3f} "
          f"KS_p={r['ks_p']:.2e}  meanNDCG={r['mean_ndcg']:.3f}  shift={r['mean_shift_se']:.1f}SE")
sd = silent_decay(corpus)
print(f"\nsilent decay (sigma={sd['sigma']}): mean shift={sd['mean_shift_se']:.2f}SE  "
      f"unpaired CIs overlap={sd['unpaired_overlap']}  paired_p={sd['paired_p']:.2e}  KS_p={sd['ks_p']:.2e}")
iv = input_vs_outcome_drift(corpus)
print(f"input vs outcome: covariate input PSI={iv['input_psi_covariate']:.3f} (fires)  "
      f"paired outcome under covariate={iv['outcome_paired_covariate_mean']:.3f} (no decay)  "
      f"paired outcome under decay={iv['outcome_paired_decay_mean']:.3f} (p={iv['decay_t_p']:.2e})")

## 5. Verification — every claim is a test

Running the module's harness confirms each pedagogical claim: pairing reduces variance, the paired test
separates sooner than the overlap read, permutation ≈ t-test, the multiple-comparison effect, raw
miscalibration and the recalibration drop, Platt's exact ranking invariance, the drift null is silent,
PSI = symmetrized KL, and the KS hand-implementation matches scipy.

In [ ]:
S.test_pairing_reduces_variance()
S.test_paired_separates_sooner_than_overlap()
S.test_ttest_rel_twin()
S.test_permutation_approximates_t()
S.test_multiple_comparison_changes_verdict()
S.test_map_pair_significant_ndcg_pair_not()
S.test_recalibration_lowers_ece()
S.test_platt_preserves_ranking_exactly()
S.test_drift_null_silent()
S.test_psi_ks_monotone_in_knob()
S.test_silent_decay_paired_beats_unpaired()
S.test_input_vs_outcome_drift()
S.test_psi_is_symmetrized_kl()
S.test_ks_hand_matches_scipy()
print("all checks passed")